# ACE-Step 1.5 VAE Round-Trip: Measuring the Codec Ceiling

Encode → decode your own tracks through the exact VAE ACE-Step uses — **no generation involved**.
What comes out is the best ACE-Step could *ever* sound for that material. Everything worse than
this in real generations is the sampler/prior's fault (LoRA-addressable); everything already lost
here is the codec ceiling (LoRA cannot buy it back).

**The codec:** `AutoencoderOobleck` (Stable Audio architecture, 169M params) — 48 kHz stereo,
strides `[2,4,4,6,10]` = **1920× downsampling → 25 Hz latent (40 ms frames)**. At 128 BPM,
16th-note hats are ~117 ms apart, so each hat gets ~3 latent frames — expect transient smearing.

**Two error metrics, and why both:**
- **Time-domain null** (original minus reconstruction): the honest total residual, and you can
  *listen* to it — but it punishes phase rotation that ears partly forgive, so it *overstates*
  perceptual loss. (Smoke test on this machine: −7.1 dB null vs −12.7 dB magnitude error on the
  same segment — ~5 dB of the null is phase.)
- **Magnitude-spectral error**: phase-blind, closer to perception, but misses genuine phase
  damage (stereo image, sub alignment). Truth is between the two; your ears are the tiebreaker.

**Outputs per track** (in `../vae_roundtrip_out/`): `*_original.wav`, `*_recon_<vae>.wav`,
`*_null_<vae>.wav` (the discarded signal, audible).

In [1]:
import torch
import torchaudio.functional as taf
import soundfile as sf
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

MUSIC = Path.home() / "Documents/Code/training_music"
OUT = Path.home() / "Documents/Code/electronic-lora/vae_roundtrip_out"
OUT.mkdir(exist_ok=True)
SR = 48_000          # VAE native rate
SEG_SECONDS = 30     # analysis window per track (from track midpoint)
HOP = 1920           # VAE total stride

# Chosen to stress different codec failure modes (all WAV to avoid mp3 confounds):
TRACKS = {
    "fisher_busy_bright": "FISHER (OZ) - You Little Beauty (Extended).wav",   # dense hats/bright top: transient test
    "lorenzo_sub_bass":   "Chris Lorenzo - Pump (Extended Mix).wav",          # bass house sub: low-end phase/pressure test
    "cassian_pads_space": "Cassian - Landa (Original Mix).wav",               # pads & reverb tails: texture/space test
    "chelina_stripped":   "Chelina Manuhutu - Big G (Original Mix).wav",      # stripped percussive: stereo image/groove test
}
for k, f in TRACKS.items():
    assert (MUSIC / f).exists(), f"missing: {f}"
print(f"{len(TRACKS)} tracks ready")

device: mps
4 tracks ready


In [2]:
from diffusers import AutoencoderOobleck

VAE_CHOICES = {
    "official": dict(pretrained_model_name_or_path="ACE-Step/Ace-Step1.5", subfolder="vae"),
    "scragvae": dict(pretrained_model_name_or_path="scragnog/Ace-Step-1.5-ScragVAE"),
}

def load_vae(name):
    vae = AutoencoderOobleck.from_pretrained(**VAE_CHOICES[name])
    return vae.to(device).eval()

vae = load_vae("official")
print(f"official VAE: {sum(p.numel() for p in vae.parameters())/1e6:.0f}M params")

/Users/zynarai/Documents/Code/electronic-lora/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/zynarai/Documents/Code/electronic-lora/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/Users/zynarai/Documents/Code/electronic-lora/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


official VAE: 169M params


In [3]:
def load_segment(path, seconds=SEG_SECONDS):
    """Middle `seconds` of the track, stereo float32 at 48 kHz. -> [2, T]"""
    info = sf.info(str(path))
    n = int(seconds * info.samplerate)
    data, sr = sf.read(str(path), start=max(0, info.frames // 2 - n // 2),
                       frames=n, dtype="float32")
    wav = torch.from_numpy(data.T if data.ndim == 2 else data[None])
    if wav.shape[0] == 1:
        wav = wav.repeat(2, 1)
    if sr != SR:
        wav = taf.resample(wav, sr, SR)
    return wav[:, : wav.shape[1] - wav.shape[1] % HOP]

def save_wav(path, wav):
    sf.write(str(path), wav.clamp(-1, 1).T.numpy(), SR)

@torch.no_grad()
def roundtrip(wav):
    """[2, T] -> deterministic reconstruction via encode->decode (10 s chunks for MPS)."""
    chunks, step = [], SR * 10
    for start in range(0, wav.shape[1], step):
        x = wav[:, start : start + step][None].to(device)
        z = vae.encode(x).latent_dist.mode()      # deterministic: measure the ceiling, not sampler noise
        chunks.append(vae.decode(z).sample[0].cpu())
    recon = torch.cat(chunks, dim=1)
    return recon[:, : wav.shape[1]]

def run_all(vae_name):
    results = {}
    for key, fname in TRACKS.items():
        orig = load_segment(MUSIC / fname)
        recon = roundtrip(orig)
        null = orig - recon
        save_wav(OUT / f"{key}_original.wav", orig)
        save_wav(OUT / f"{key}_recon_{vae_name}.wav", recon)
        save_wav(OUT / f"{key}_null_{vae_name}.wav", null)
        results[key] = (orig, recon, null)
        print(f"{key}: done")
    return results

results = run_all("official")

fisher_busy_bright: done
lorenzo_sub_bass: done
cassian_pads_space: done
chelina_stripped: done


## Metrics

Rough interpretation guide for the **time-domain null**: −30 dB or quieter ≈ near-transparent;
−20 dB = clearly audible degradation; −10 dB = heavy loss. Expect the magnitude-spectral error
to read ~5 dB better (phase-blind). Banded columns split the null by frequency range —
expect >6 kHz (transients) to be worst.

In [ ]:
import numpy as np

W = torch.hann_window(2048)

def stft_mag(x):
    return torch.stft(x.mean(0), 2048, hop_length=512, window=W, return_complex=True).abs()

def null_db(orig, null):
    return (20 * torch.log10(null.pow(2).mean().sqrt() / orig.pow(2).mean().sqrt())).item()

def spec_mag_db(orig, recon):
    So, Sr = stft_mag(orig), stft_mag(recon)
    return (10 * torch.log10(((So - Sr) ** 2).mean() / (So ** 2).mean())).item()

def band_db(orig, null, lo=None, hi=None):
    freqs = torch.from_numpy(np.fft.rfftfreq(2048, 1 / SR))
    sel = torch.ones_like(freqs, dtype=torch.bool)
    if lo: sel &= freqs >= lo
    if hi: sel &= freqs < hi
    p = lambda x: (stft_mag(x)[sel] ** 2).mean().item()
    return 10 * np.log10(p(null) / (p(orig) + 1e-12) + 1e-12)

print(f"{'track':22s} {'null':>7s} {'magspec':>8s} {'sub<120':>8s} {'mids':>7s} {'>6kHz':>7s}")
for key, (orig, recon, null) in results.items():
    print(f"{key:22s} {null_db(orig, null):7.1f} {spec_mag_db(orig, recon):8.1f} "
          f"{band_db(orig, null, None, 120):8.1f} {band_db(orig, null, 120, 6000):7.1f} "
          f"{band_db(orig, null, 6000, None):7.1f}")

In [ ]:
import matplotlib.pyplot as plt

def specshow(ax, x, title):
    S = stft_mag(x)
    ax.imshow(20 * torch.log10(S + 1e-6).numpy(), origin="lower", aspect="auto",
              cmap="magma", vmin=-60, vmax=40, extent=[0, SEG_SECONDS, 0, SR / 2 / 1000])
    ax.set_title(title); ax.set_ylabel("kHz")

for key, (orig, recon, null) in results.items():
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    specshow(axes[0], orig, f"{key} — original")
    specshow(axes[1], recon, "reconstruction")
    specshow(axes[2], null, "null (what the codec discards)")
    axes[2].set_xlabel("seconds")
    plt.tight_layout(); plt.show()

## Optional: A/B against ScragVAE

Community VAE finetune advertised for transient handling / timbre. Writes
`*_recon_scragvae.wav` / `*_null_scragvae.wav` next to the official ones.

In [ ]:
vae = load_vae("scragvae")
results_scrag = run_all("scragvae")
print()
print(f"{'track':22s} {'official':>9s} {'scragvae':>9s}   (time-domain null dB; more negative = better)")
for key in TRACKS:
    orig, _, n_off = results[key]
    _, _, n_scr = results_scrag[key]
    print(f"{key:22s} {null_db(orig, n_off):9.1f} {null_db(orig, n_scr):9.1f}")

## Listening guide

Open `vae_roundtrip_out/` and A/B on your monitors:

1. **Original vs recon, blind if you can.** If they're hard to tell apart, the ceiling is high
   → generation badness is mostly sampler/prior → LoRA + more inference steps have real headroom.
2. **Play the nulls loud** — literally the discarded signal. Listen for hat/cymbal transients
   (expected worst), sub ghost notes, vocal-chop consonants, stereo-width artifacts. *Caveat:*
   some null content is phase rotation, which A/B ears partly forgive — judge degradation from
   the A/B, use the null to localize *where* it lives.
3. **FISHER vs Cassian:** if the busy/bright track nulls much louder than the pads track,
   that's time-compression (transient) loss, not bandwidth loss — consistent with a 25 Hz latent.
4. **For the LoRA project:** what survives the round-trip is the budget your LoRA competes for.
   What dies here needs a better VAE (cell above) or a different model family — not better
   training data.